# data_processor_gru_chang.py — Test Notebook

Tests the **GRU** data pipeline (`qwenvl/data/data_processor_gru_chang.py`) end to end:
`_build_messages` → `__getitem__` (tokenization + `<gru>`/GRU feature build) →
`FlattenedDataCollator` (batching, GRU stacking).

Focus vs. the plain `data_processor`: every nav sample gets **one** `<gru>` placeholder token
whose embedding the model later overwrites with a projected trajectory vector. Each item now
also carries `gru_features (1,T,7)`, `gru_lengths`, `has_gru`, `motion_token_id`,
`motion_positions`, `motion_token_count`.

---

## Table of Contents

| Section | What it covers |
|---------|----------------|
| **1 — Setup** | sys.path, `NAVILA_BASE` + `BASE_GRU_DIR`, processor (must know `<gru>`), `LazySupervisedDataset`, dataset index map |
| **2 — GRU feature primitives** | `actions_to_motion_features`: action IDs → 7-D motion features |
| **3 — R2R (has_gru=True)** | `__getitem__` GRU outputs, `<gru>` placement check, `_build_messages`, sequence structure |
| **4 — Flattened collator** | GRU stacking: `gru_features (B,1,T,7)`, `gru_lengths (B,1)`, `has_gru (B,)` |
| **5 — Non-nav datasets** | scanqa / video_chatgpt / envdrop(legacy) / sharegpt* → `has_gru=0`, no `<gru>`, dummy GRU |
| **6 — Mixed batch** | one nav + one non-nav collated together → `has_gru=[1,0]`, per-sample `motion_positions` |

---

> **Key format facts.** GRU input = a sequence of integer action IDs
> (`STOP=0, FORWARD=1, TURN_LEFT=2, TURN_RIGHT=3`) turned into 7-D features
> `[cum_x, cum_y, sin θ, cos θ, dyaw, is_forward, is_turn]` → tensor `(T,7)`.
> Per sample this is wrapped to one slot `(1,T,7)`; the collator stacks to `(B,1,T,7)`.
> Only R2R/RxR/Human (native `q,a,frames`) and EnvDrop-motion (`frames,motion,q`) are
> `has_gru=True`; everything else is `has_gru=False` with a dummy `(1,1,7)`.

## Section 1 — Setup

Add the project root and `qwen-vl-finetune` to `sys.path` so `qwenvl.*` imports resolve from any cwd.

In [ ]:
import sys
from pathlib import Path

project_root = Path("../../..").resolve()
sys.path.insert(0, str(project_root))
sys.path.insert(0, str(project_root / "qwen-vl-finetune"))
print(f"Project root: {project_root}")

Auto-detect the dataset root (`NAVILA_BASE`) and the self-contained GRU base dir
(`BASE_GRU_DIR` = backbone + aligned projector + frozen GRU + tokenizer with `<gru>`@151669).
`MockDataArgs` mirrors `scripts/slurm_gru_qwen_chang.sh` (incl. the `gru_*` knobs the dataset reads).

> Actual annotation/data paths come from the registry in `qwenvl/data/__init__.py`; `DATASET_PATHS`
> below is only for a quick existence check. Override either root with an env var if your mounts differ.

In [ ]:
import os
from dataclasses import dataclass

# --- dataset root -----------------------------------------------------------
_NAVILA_CANDIDATES = [
    os.environ.get("NAVILA_BASE", ""),
    "/opt/IROS_proj/NaVILA-Dataset",
    "/home/rithvik/IROS_proj/NaVILA-Dataset",
    "/weka/scratch/tinoosh/iros_dataset/NaVILA-Dataset",
]
NAVILA_BASE = next((c for c in _NAVILA_CANDIDATES if c and os.path.exists(c)),
                   "/opt/IROS_proj/NaVILA-Dataset")
print(f"NAVILA_BASE: {NAVILA_BASE}  (exists={os.path.exists(NAVILA_BASE)})")

# --- GRU base dir (tokenizer must contain <gru>) ----------------------------
_BASE_GRU_CANDIDATES = [
    os.environ.get("BASE_GRU_DIR", ""),
    "/opt/IROS_proj/models_ckpts/trained/gru/base_qwen_with_gru",
    "/home/rithvik/IROS_proj/models_ckpts/trained/gru/base_qwen_with_gru",
    "/home/chang/vla/navilla/models/base_qwen_with_gru",
]
BASE_GRU_DIR = next((c for c in _BASE_GRU_CANDIDATES if c and os.path.exists(c)),
                    _BASE_GRU_CANDIDATES[1])
print(f"BASE_GRU_DIR: {BASE_GRU_DIR}  (exists={os.path.exists(BASE_GRU_DIR)})")
if not os.path.exists(BASE_GRU_DIR):
    print("  !! set BASE_GRU_DIR env var to the dir prepared by tools/prepare_base_gru_dir_chang.py")

DATASET_PATHS = {
    "r2r":           f"{NAVILA_BASE}/R2R/annotations.json",
    "human":         f"{NAVILA_BASE}/Human/annotations.json",
    "rxr":           f"{NAVILA_BASE}/RxR/annotations.json",
    "scanqa":        f"{NAVILA_BASE}/ScanQA/annotations/ScanQA_v1.0_train_reformat.json",
    "envdrop":       f"{NAVILA_BASE}/EnvDrop/annotations.json",
    "video_chatgpt": f"{NAVILA_BASE}/Video-ChatGPT/VideoInstruct100K.json",
    "sharegptvideo": f"{NAVILA_BASE}/ShareGPTVideo/video_caption_300k.jsonl",
    "sharegpt4v":    f"{NAVILA_BASE}/ShareGPT4V/sharegpt4v_mix665k_cap23k_coco-ap9k_lcs3k_sam9k_div2k.json",
}

# All 8 datasets, to exercise both has_gru=True (r2r/human/rxr) and has_gru=False branches.
DATASET_USE = "r2r,human,rxr,scanqa,envdrop,video_chatgpt,sharegptvideo,sharegpt4v"
MODEL_TYPE  = "qwen3vl"  # "qwen2vl" | "qwen2.5vl" | "qwen3vl"

print("\nAvailable datasets:")
for name, path in DATASET_PATHS.items():
    marker = "OK     " if os.path.exists(path) else "MISSING"
    active = "  <-- selected" if name in DATASET_USE.split(",") else ""
    print(f"  [{marker}] {name:15s}  {path}{active}")

@dataclass
class MockDataArgs:
    dataset_use: str = DATASET_USE
    data_flatten: bool = False
    data_packing: bool = False
    model_type: str = MODEL_TYPE
    max_pixels: int = 28 * 28 * 576
    min_pixels: int = 28 * 28 * 16
    video_max_frames: int = 8
    video_min_frames: int = 4
    video_max_pixels: int = 1024 * 28 * 28
    video_min_pixels: int = 256 * 28 * 28
    video_max_total_pixels: int = 1664 * 28 * 28
    video_min_total_pixels: int = 256 * 28 * 28
    video_fps: float = 2.0
    # GRU knobs (read by LazySupervisedDataset / _get_item)
    gru_history_slots: int = 8
    gru_min_seq_len: int = 1
    motion_token_text: str = "<gru>"   # informational; processor pins to DEFAULT_MOTION_TOKEN
    base_interval: int = 2

data_args = MockDataArgs()
print(f"\ndataset_use: {data_args.dataset_use}")
print(f"model_type:  {data_args.model_type}")

Load `AutoProcessor` from `BASE_GRU_DIR`. **Critical:** its tokenizer must know the `<gru>`
special token (id 151669), otherwise the injection position can't be located. We assert that here.

In [ ]:
from transformers import AutoProcessor
from qwenvl.data.data_processor_gru_chang import DEFAULT_MOTION_TOKEN

processor = AutoProcessor.from_pretrained(BASE_GRU_DIR)
tokenizer = processor.tokenizer
print(f"Tokenizer vocab size : {tokenizer.vocab_size}")
print(f"Image processor type : {type(processor.image_processor).__name__}")

gid = tokenizer.convert_tokens_to_ids(DEFAULT_MOTION_TOKEN)
print(f"\n{DEFAULT_MOTION_TOKEN} -> id {gid}")
assert gid is not None and gid != tokenizer.unk_token_id, (
    f"{DEFAULT_MOTION_TOKEN} is not a known token in this tokenizer — "
    f"point BASE_GRU_DIR at the dir prepared by tools/prepare_base_gru_dir_chang.py"
)
# round-trip: a single <gru> must tokenize to exactly one id
enc = tokenizer(DEFAULT_MOTION_TOKEN, add_special_tokens=False)["input_ids"]
print(f"round-trip ids       : {enc}  (expect a single id == {gid})")

Instantiate `LazySupervisedDataset` (from `data_processor_gru_chang`). Annotations are read into
`list_data_dict` and the per-trajectory cumulative action index is built; image/GRU work is deferred
to `__getitem__`.

In [ ]:
from qwenvl.data.data_processor_gru_chang import LazySupervisedDataset

dataset = LazySupervisedDataset(processor, data_args)
print(f"Total samples           : {len(dataset)}")
print(f"Trajectories indexed     : {len(dataset.traj_cumulative_actions)} (for native GRU features)")
print(f"gru_history_slots        : {dataset.gru_history_slots}")

In [ ]:
DS_START = {}
for i, ann in enumerate(dataset.list_data_dict):
    dp = ann.get("data_path", "")
    for tag in ("R2R", "Human", "RxR", "ScanQA", "EnvDrop",
                "Video-ChatGPT", "ShareGPTVideo", "ShareGPT4V"):
        key = tag.lower().replace("-", "")
        if tag in dp and key not in DS_START:
            DS_START[key] = i
    if len(DS_START) == 8:
        break

print("Dataset start indices:")
for k, v in DS_START.items():
    print(f"  {k:20s} idx={v:>9d}  keys={list(dataset.list_data_dict[v].keys())}")

## Section 2 — GRU feature primitives

`actions_to_motion_features` is the contract the GRU consumes: a list of integer action IDs becomes a
`(T, 7)` float tensor. Columns: `[cum_x, cum_y, sin θ, cos θ, dyaw, is_forward, is_turn]`
(defaults: forward step 0.25 m, turn 15°).

In [ ]:
import torch
from qwenvl.data.data_processor_gru_chang import (
    actions_to_motion_features, STOP, FORWARD, TURN_LEFT, TURN_RIGHT,
)

demo_actions = [FORWARD, FORWARD, TURN_LEFT, FORWARD, TURN_RIGHT, STOP]
feats = actions_to_motion_features(demo_actions)
print(f"actions {demo_actions} -> features {tuple(feats.shape)} (T, 7)\n")
cols = ["cum_x", "cum_y", "sinθ", "cosθ", "dyaw", "is_fwd", "is_turn"]
print("  " + "  ".join(f"{c:>7s}" for c in cols))
for a, row in zip(demo_actions, feats):
    print(f"a={a}  " + "  ".join(f"{x:7.3f}" for x in row.tolist()))

## Section 3 — R2R (has_gru = True)

`dataset[INDEX]` runs the full GRU pipeline. Beyond the usual tensors it returns the GRU block:
`gru_features (1,T,7)`, `gru_lengths (1,)`, `has_gru`, `motion_token_id`, `motion_positions`,
`motion_token_count`. We then verify the `<gru>` token actually sits at `motion_positions` in
`input_ids`, and that exactly one is emitted.

In [ ]:
import numpy as np

# ── change INDEX to inspect a different nav sample ──────────────────────────
INDEX = DS_START.get("r2r", 0)
# ────────────────────────────────────────────────────────────────────────────

sample = dataset[INDEX]

print("=== __getitem__ output ===\n")
for k, v in sample.items():
    if hasattr(v, "shape"):
        print(f"  {k:20s} shape={list(v.shape)}  dtype={v.dtype}")
    else:
        print(f"  {k:20s} {v}")

print("\n=== GRU block ===")
print(f"  has_gru            : {int(sample['has_gru'])}")
print(f"  gru_features       : {list(sample['gru_features'].shape)}  (slots=1, T, 7)")
print(f"  gru_lengths        : {sample['gru_lengths'].tolist()}  (valid steps per slot)")
print(f"  motion_token_id    : {int(sample['motion_token_id'])}")
print(f"  motion_token_count : {int(sample['motion_token_count'])}")
print(f"  motion_positions   : {sample['motion_positions'].tolist()}")

# --- verify <gru> placement -------------------------------------------------
ids = sample["input_ids"][0]
mtid = int(sample["motion_token_id"])
pos  = sample["motion_positions"].tolist()
assert int(sample["has_gru"]) == 1, "R2R sample should be has_gru=1"
assert int(sample["motion_token_count"]) == 1, "expect exactly one <gru> per nav sample"
assert all(int(ids[p]) == mtid for p in pos), "motion_positions must point at <gru> tokens"
print("\nOK: exactly one <gru>, and input_ids at motion_positions == motion_token_id")

# labels coverage
labels = sample["labels"]
total, active = labels.numel(), (labels != -100).sum().item()
print(f"labels active      : {active}/{total} ({active/total:.1%})")

`_build_messages` on the same annotation: note the nav prompt now **prepends**
`"This is the encoded trajectory so far: <gru>"` before the navigation prompt, and returns
`has_gru=True`.

In [ ]:
from qwenvl.data.data_processor_gru_chang import _build_messages

ann = dataset.list_data_dict[INDEX]
# mirror what _get_item passes through (history-slot count)
patched = dict(ann); patched["_gru_history_slots"] = dataset.gru_history_slots
messages, has_gru = _build_messages(patched, Path(ann["data_path"]))

print(f"has_gru: {has_gru}  |  turns: {len(messages)}\n")
for turn in messages:
    print(f"[{turn['role']}]")
    for c in turn["content"]:
        if c["type"] == "text":
            print(f"  text : {c['text'][:120]}")
        else:
            img = c["image"]
            shown = img if isinstance(img, str) else f"{img.size} {img.mode} {type(img).__name__}"
            print(f"  image: {shown}")
    print()

Sequence-structure view: text collapsed, images shown as patch counts, and the single `<gru>`
placeholder marked explicitly so you can see where the trajectory vector gets injected.

In [ ]:
def show_structure(input_ids, tok):
    IMAGE_PAD = tok.convert_tokens_to_ids("<|image_pad|>")
    VIS_START = tok.convert_tokens_to_ids("<|vision_start|>")
    IM_START  = tok.convert_tokens_to_ids("<|im_start|>")
    IM_END    = tok.convert_tokens_to_ids("<|im_end|>")
    GRU       = tok.convert_tokens_to_ids(DEFAULT_MOTION_TOKEN)
    specials  = (VIS_START, IMAGE_PAD, IM_START, IM_END, GRU)
    segs, i = [], 0
    while i < len(input_ids):
        tid = input_ids[i]
        if tid == VIS_START:
            j = i + 1
            while j < len(input_ids) and input_ids[j] == IMAGE_PAD:
                j += 1
            segs.append(f"[IMAGE: {j - i - 1} patches]"); i = j + 1
        elif tid == GRU:
            segs.append(f"[<gru> @ {i}]"); i += 1
        elif tid in (IM_START, IM_END):
            segs.append(tok.decode([tid])); i += 1
        else:
            j = i
            while j < len(input_ids) and input_ids[j] not in specials:
                j += 1
            text = tok.decode(input_ids[i:j], skip_special_tokens=True).strip()
            if text:
                segs.append(f'"{text[:200]}"')
            i = j
    return segs

print("--- R2R sequence structure ---")
for s in show_structure(sample["input_ids"][0].tolist(), tokenizer):
    print(" ", s)

## Section 4 — Flattened collator (GRU stacking)

`FlattenedDataCollatorForSupervisedDataset` packs samples and stacks the GRU block:
`gru_features (B,1,T,7)` (T padded across the batch), `gru_lengths (B,1)`, `has_gru (B,)`,
plus `motion_token_id`, `motion_token_count`, and per-sample `motion_positions`.

In [ ]:
from qwenvl.data.data_processor_gru_chang import FlattenedDataCollatorForSupervisedDataset

instances = [dataset[INDEX], dataset[INDEX + 1]]
collator = FlattenedDataCollatorForSupervisedDataset(tokenizer=tokenizer)
batch = collator(instances)

print("=== batch (flattened) ===\n")
for k, v in batch.items():
    if v is None:
        print(f"  {k:20s} None")
    elif hasattr(v, "shape"):
        print(f"  {k:20s} shape={list(v.shape)}  dtype={v.dtype}")
    else:
        print(f"  {k:20s} {v}")

print(f"\nhas_gru            : {batch['has_gru'].tolist()}")
print(f"gru_features        : {list(batch['gru_features'].shape)}  (B, slots=1, max_T, 7)")
print(f"gru_lengths         : {batch['gru_lengths'].tolist()}")
print(f"motion_token_id     : {batch['motion_token_id']}")
print(f"motion_token_count  : {batch['motion_token_count'].tolist()}")
print(f"motion_positions    : {batch['motion_positions']}  (per-sample token indices)")

## Section 5 — Non-nav datasets (has_gru = False)

ScanQA / Video-ChatGPT / EnvDrop(legacy `{video_id, instruction}`) / ShareGPTVideo / ShareGPT4V do
**not** carry a per-step action stream. `_build_messages` returns `has_gru=False`, emits **no** `<gru>`
token, and `_get_item` attaches a dummy `gru_features (1,1,7)` with `gru_lengths=[0]` purely so the
collator can stack. We confirm: `has_gru=0`, `motion_token_count=0`, dummy GRU shape.

In [ ]:
checks = ["scanqa", "videochatgpt", "envdrop", "sharegptvideo", "sharegpt4v"]
for key in checks:
    if key not in DS_START:
        print(f"[skip] {key}: not loaded"); continue
    idx = DS_START[key]
    try:
        item = dataset[idx]
    except Exception as e:
        print(f"[skip] {key} (idx={idx}): {e}"); continue
    hg   = int(item["has_gru"])
    cnt  = int(item["motion_token_count"])
    gshp = list(item["gru_features"].shape)
    glen = item["gru_lengths"].tolist()
    ok   = (hg == 0 and cnt == 0)
    print(f"{key:14s} idx={idx:>9d}  has_gru={hg}  motion_token_count={cnt}  "
          f"gru_features={gshp}  gru_lengths={glen}  {'OK' if ok else 'UNEXPECTED'}")

## Section 6 — Mixed batch (nav + non-nav)

The real training batches mix `has_gru=1` and `has_gru=0` rows. Collate one R2R sample with one
ScanQA sample: `has_gru=[1,0]`, `gru_features` padded to a common `T`, and `motion_positions` shows
the `<gru>` index only for the nav row (empty list for the QA row). At forward time the model's
`_inject_gru` overwrites the `<gru>` embedding only on `has_gru=1` rows.

In [ ]:
nav_idx = DS_START.get("r2r", 0)
qa_key  = next((k for k in ("scanqa", "videochatgpt", "sharegpt4v") if k in DS_START), None)
qa_idx  = DS_START[qa_key]
print(f"nav sample : r2r idx={nav_idx}")
print(f"qa  sample : {qa_key} idx={qa_idx}\n")

mixed = collator([dataset[nav_idx], dataset[qa_idx]])
print(f"has_gru           : {mixed['has_gru'].tolist()}   (expect [1, 0])")
print(f"gru_features       : {list(mixed['gru_features'].shape)}")
print(f"gru_lengths        : {mixed['gru_lengths'].tolist()}")
print(f"motion_token_count : {mixed['motion_token_count'].tolist()}")
print(f"motion_positions   : {mixed['motion_positions']}")

assert mixed["has_gru"].tolist() == [1, 0], "mixed batch should be [nav=1, qa=0]"
print("\nOK: nav row carries one <gru>; qa row carries none.")